In [ ]:
pip install numpy pandas matplotlib seaborn scikit-learn tensorflow lime

In [ ]:
# if tensorflow install shows err, then check ur python version

!python --version

# Recommended Python version for TensorFlow:
# Python 3.10
# Python 3.11

## EXP 1 — Hidden Markov Model (HMM) for Weather Prediction

Aim:
To design and implement a Hidden Markov Model (HMM) for weather prediction using observable daily activities.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix

# Hidden States (Weather)
states = ["Sunny", "Rainy", "Cloudy"]

# Observable Activities
observations = ["Umbrella", "Sunglasses", "Jackets", "Normal"]

# Initial probabilities
start_prob = np.array([0.4, 0.3, 0.3])

# Transition Probability Matrix
transition_prob = np.array([
    [0.6, 0.2, 0.2],
    [0.3, 0.5, 0.2],
    [0.4, 0.3, 0.3]
])

# Emission Probability Matrix
emission_prob = np.array([
    [0.1, 0.6, 0.1, 0.2],
    [0.7, 0.05, 0.2, 0.05],
    [0.3, 0.2, 0.4, 0.1]
])

obs_sequence = [0, 2, 1]

def forward_algorithm(obs_seq):
    N = len(states)
    T = len(obs_seq)

    alpha = np.zeros((N, T))

    for i in range(N):
        alpha[i, 0] = start_prob[i] * emission_prob[i, obs_seq[0]]

    for t in range(1, T):
        for j in range(N):
            alpha[j, t] = np.sum(
                alpha[:, t-1] * transition_prob[:, j]
            ) * emission_prob[j, obs_seq[t]]

    probability = np.sum(alpha[:, T-1])

    return alpha, probability

alpha_matrix, sequence_probability = forward_algorithm(obs_sequence)

print(alpha_matrix)
print(sequence_probability)

def viterbi(obs_seq):
    n_states = len(states)
    T = len(obs_seq)

    dp = np.zeros((n_states, T))
    backpointer = np.zeros((n_states, T), dtype=int)

    dp[:, 0] = start_prob * emission_prob[:, obs_seq[0]]

    for t in range(1, T):
        for s in range(n_states):
            prob = dp[:, t-1] * transition_prob[:, s]
            dp[s, t] = np.max(prob) * emission_prob[s, obs_seq[t]]
            backpointer[s, t] = np.argmax(prob)

    best_path = [np.argmax(dp[:, T-1])]

    for t in range(T-1, 0, -1):
        best_path.insert(0, backpointer[best_path[0], t])

    return [states[i] for i in best_path]

predicted_states = viterbi(obs_sequence)

print(predicted_states)

## EXP 2 — Bayesian Network for Student Pass Prediction

Aim:
To design and implement a Bayesian Network for predicting whether a student will pass a final exam.

In [ ]:
from pgmpy.models import BayesianModel
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# Define structure
model = BayesianModel([
    ('StudyHours', 'InternalMarks'),
    ('Attendance', 'InternalMarks'),
    ('InternalMarks', 'Pass')
])

# Define CPDs
cpd_SH = TabularCPD(variable='StudyHours', variable_card=2, values=[[0.4], [0.6]])
cpd_AT = TabularCPD(variable='Attendance', variable_card=2, values=[[0.3], [0.7]])

cpd_IM = TabularCPD(
    variable='InternalMarks', variable_card=2,
    values=[[0.1, 0.3, 0.4, 0.8],
            [0.9, 0.7, 0.6, 0.2]],
    evidence=['StudyHours', 'Attendance'],
    evidence_card=[2, 2]
)

cpd_P = TabularCPD(
    variable='Pass', variable_card=2,
    values=[[0.05, 0.6],
            [0.95, 0.4]],
    evidence=['InternalMarks'],
    evidence_card=[2]
)

# Add CPDs
model.add_cpds(cpd_SH, cpd_AT, cpd_IM, cpd_P)

# Inference
infer = VariableElimination(model)

# Query example
result = infer.query(variables=['Pass'], evidence={'StudyHours':1, 'Attendance':1})
print(result)

## EXP 3 — Gaussian Mixture Model (GMM)

Aim:
To design and implement a Gaussian Mixture Model (GMM) for outcome prediction.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score

np.random.seed(42)

X1 = np.random.multivariate_normal(
    mean=[-2,-2],
    cov=[[0.5,0.1],[0.1,0.5]],
    size=150
)

X2 = np.random.multivariate_normal(
    mean=[2,-2],
    cov=[[0.6,-0.2],[-0.2,0.6]],
    size=150
)

X3 = np.random.multivariate_normal(
    mean=[2,2],
    cov=[[0.4,0.0],[0.0,0.4]],
    size=150
)

X = np.vstack((X1,X2,X3))

true_labels = np.array([0]*150+[1]*150+[2]*150)

gmm = GaussianMixture(
    n_components=3,
    covariance_type='full',
    random_state=42
)

gmm.fit(X)

labels = gmm.predict(X)

ari = adjusted_rand_score(true_labels, labels)

print("Adjusted Rand Index:", ari)

## EXP 4 — Generative Multi-Layer Network Model

Aim:
To build and train a Generative Multi-Layer Network Model.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense
import matplotlib.pyplot as plt

np.random.seed(42)

num_samples = 5000
latent_dim = 5

z = np.random.normal(0, 1, (num_samples, latent_dim))

Y = np.zeros((num_samples,2))
Y[:,0] = np.sin(z[:,0])
Y[:,1] = np.cos(z[:,1])

model = Sequential([
    Input(shape=(latent_dim,)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(2)
])

model.compile(
    optimizer='adam',
    loss='mse'
)

history = model.fit(
    z,
    Y,
    epochs=100,
    batch_size=32,
    verbose=1
)

z_new = np.random.normal(0,1,(1000,latent_dim))

generated = model.predict(z_new)

plt.scatter(generated[:,0], generated[:,1])
plt.show()

## EXP 5 — DCGAN for Image Generation

Aim:
To build and train a Deep Convolution GAN (DCGAN).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Reshape, Flatten
from tensorflow.keras.layers import Conv2D, Conv2DTranspose
from tensorflow.keras.layers import LeakyReLU, Dropout
from tensorflow.keras.optimizers import Adam

(X_train, _), (_, _) = mnist.load_data()

X_train = (X_train.astype('float32') - 127.5) / 127.5
X_train = np.expand_dims(X_train, axis=-1)

generator = Sequential([
    Dense(7 * 7 * 128, input_dim=100),
    LeakyReLU(0.2),
    Reshape((7, 7, 128)),
    Conv2DTranspose(64, kernel_size=4, strides=2, padding='same'),
    LeakyReLU(0.2),
    Conv2DTranspose(
        1,
        kernel_size=4,
        strides=2,
        padding='same',
        activation='tanh'
    )
])

discriminator = Sequential([
    Conv2D(
        64,
        kernel_size=4,
        strides=2,
        padding='same',
        input_shape=(28,28,1)
    ),
    LeakyReLU(0.2),
    Dropout(0.3),
    Flatten(),
    Dense(1, activation='sigmoid')
])

discriminator.compile(
    optimizer=Adam(0.0002),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## EXP 6 — Transfer Learning using VGG16

Aim:
To explore transfer learning using VGG16.

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

x_train = tf.image.resize(x_train, (48,48))
x_test = tf.image.resize(x_test, (48,48))

x_train = x_train / 255.0
x_test = x_test / 255.0

base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(48,48,3)
)

for layer in base_model.layers:
    layer.trainable = False

model = Sequential([
    base_model,
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    x_train,
    y_train,
    epochs=2,
    validation_split=0.2,
    batch_size=64
)

## EXP 7 — LIME Model Interpretation

Aim:
To implement and analyze the LIME framework.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from lime import lime_tabular

iris = load_iris()

X = iris.data
y = iris.target

feature_names = iris.feature_names
class_names = iris.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)

print("Model Accuracy:", accuracy)

explainer = lime_tabular.LimeTabularExplainer(
    training_data=X_train,
    feature_names=feature_names,
    class_names=class_names,
    mode='classification'
)

sample = X_test[0]

explanation = explainer.explain_instance(
    data_row=sample,
    predict_fn=model.predict_proba,
    num_features=4
)

for feature, weight in explanation.as_list():
    print(feature, weight)

fig = explanation.as_pyplot_figure()
plt.show()

EXPERIMENT 1
Hidden Markov Model (HMM) for Weather Prediction
Aim

To design and implement a Hidden Markov Model (HMM) for weather prediction using observable daily activities.

Introduction

A Hidden Markov Model (HMM) is a statistical machine learning model used for sequence prediction problems where the system states are hidden but outputs are observable.

In this experiment, weather conditions such as Sunny, Rainy, and Cloudy are hidden states, while observable activities such as carrying umbrellas or wearing sunglasses are visible outputs.

HMM predicts the most probable hidden state sequence using probability distributions.

Definition

A Hidden Markov Model is a probabilistic model in which:

The system follows the Markov property.
Current state depends only on previous state.
States are hidden.
Outputs are observable.
Components of HMM
Hidden States
Observable States
Initial Probability
Transition Probability
Emission Probability
Algorithms Used
1. Forward Algorithm

Calculates probability of observation sequence.

2. Viterbi Algorithm

Finds most likely hidden state sequence.

Types of HMM
Type	Description
Discrete HMM	Observations are discrete values
Continuous HMM	Observations are continuous values
Left-to-Right HMM	State progression only forward
Ergodic HMM	Any state can transition to any state
Difference Between Forward and Viterbi Algorithm
Forward Algorithm	Viterbi Algorithm
Calculates total probability	Finds best path
Used for evaluation	Used for decoding
Sums probabilities	Takes maximum probability
Applications
Speech Recognition
Weather Prediction
NLP
Bioinformatics
Activity Recognition
Advantages
Handles sequential data well
Works with hidden information
Probabilistic approach
Limitations
Assumes Markov property
Computationally expensive for large states
Requires probability estimation
Viva Questions
What is Markov Property?

Current state depends only on previous state.

What are hidden states?

States that cannot be directly observed.

What is emission probability?

Probability of observable output from hidden state.

Why use Viterbi algorithm?

To find most probable hidden sequence.

EXPERIMENT 2
Bayesian Network for Student Pass Prediction
Aim

To design and implement a Bayesian Network for predicting whether a student will pass a final exam.

Introduction

Bayesian Networks are probabilistic graphical models that represent relationships between variables using directed acyclic graphs.

This experiment predicts student performance using:

Study Hours
Attendance
Internal Marks
Definition

A Bayesian Network is a graphical probabilistic model representing conditional dependencies among variables.

Key Concepts
Nodes represent variables.
Edges represent dependencies.
Probability tables define relationships.
Uses Bayes theorem.
Bayes Theorem

genui{"math_block_widget_always_prefetch_v2":{"content":"P(A|B)=\frac{P(B|A)P(A)}{P(B)}"}}

Types of Bayesian Networks
Type	Description
Naive Bayes	Assumes feature independence
Dynamic Bayesian Network	Works on sequential data
Gaussian Bayesian Network	Continuous variables
Hybrid Bayesian Network	Both continuous and discrete
Difference Between Bayesian Network and Naive Bayes
Bayesian Network	Naive Bayes
Handles dependencies	Assumes independence
Complex structure	Simple structure
More accurate	Faster computation
Applications
Medical Diagnosis
Student Performance Prediction
Spam Detection
Risk Analysis
Recommendation Systems
Advantages
Handles uncertainty
Probabilistic predictions
Easy interpretation
Limitations
Complex for large datasets
Probability estimation difficult
Requires domain knowledge
Viva Questions
What is Bayes theorem?

It calculates conditional probability.

What is conditional probability?

Probability of an event occurring given another event occurred.

What is Naive Bayes?

Classifier assuming all features are independent.

EXPERIMENT 3
Gaussian Mixture Model (GMM)
Aim

To design and implement a Gaussian Mixture Model for outcome prediction.

Introduction

Gaussian Mixture Model is a probabilistic clustering algorithm that assumes data points are generated from multiple Gaussian distributions.

Unlike K-Means, GMM provides soft clustering.

Definition

A Gaussian Mixture Model is a weighted combination of multiple Gaussian distributions used for clustering and density estimation.

Gaussian Distribution Formula

genui{"math_block_widget_always_prefetch_v2":{"content":"f(x)=\frac{1}{\sqrt{2\pi\sigma^2}}e^{-\frac{(x-\mu)^2}{2\sigma^2}}"}}

Key Concepts
Mean
Covariance
Mixture Components
Probability Distribution
Expectation Maximization (EM)
Types of Clustering
Type	Description
Hard Clustering	One cluster only
Soft Clustering	Probability-based membership

GMM uses soft clustering.

Difference Between GMM and K-Means
GMM	K-Means
Soft clustering	Hard clustering
Uses probability	Uses distance
Handles covariance	Assumes spherical clusters
More flexible	Faster
Applications
Image Segmentation
Pattern Recognition
Speech Recognition
Anomaly Detection
Customer Segmentation
Advantages
Flexible clustering
Handles overlapping clusters
Probability outputs
Limitations
Computationally expensive
Sensitive to initialization
Overfitting possible
Viva Questions
What is covariance?

Measure of relation between variables.

What is soft clustering?

A point can belong to multiple clusters with probabilities.

What is EM algorithm?

Optimization algorithm used in GMM.

EXPERIMENT 4
Generative Multi-Layer Network Model
Aim

To build and train a Generative Multi-Layer Network Model.

Introduction

Generative neural networks learn patterns from training data and generate new data samples.

This experiment uses a feedforward neural network (MLP) to map latent variables into output distributions.

Definition

A Generative Multi-Layer Network is a neural network capable of learning data distribution and generating similar outputs.

Components
Input Layer
Hidden Layers
Output Layer
Activation Functions
Optimizer
Types of Neural Networks
Type	Description
Feedforward Neural Network	One-direction flow
CNN	Image processing
RNN	Sequential data
GAN	Data generation
Difference Between Discriminative and Generative Models
Generative Model	Discriminative Model
Generates data	Classifies data
Learns distribution	Learns boundaries
Example: GAN	Example: Logistic Regression
Applications
Image Generation
Speech Synthesis
Music Generation
Simulation
AI Content Creation
Advantages
Learns complex patterns
Generates synthetic data
Useful in simulations
Limitations
Requires large data
Computationally expensive
Training instability
Viva Questions
What is latent space?

Compressed representation of data.

What is activation function?

Function introducing non-linearity.

Why use ReLU?

Faster and avoids vanishing gradient.

EXPERIMENT 5
Deep Convolutional Generative Adversarial Network (DCGAN)
Aim

To build and train a Deep Convolution GAN for image generation.

Introduction

DCGAN combines Convolutional Neural Networks with GAN architecture for realistic image generation.

It contains two neural networks:

Generator
Discriminator

Both compete against each other.

Definition

A DCGAN is a deep learning generative model using convolutional layers to generate realistic images.

GAN Objective Function

genui{"math_block_widget_always_prefetch_v2":{"content":"\min_G\max_DV(D,G)=\mathbb{E}{x\sim p{data}(x)}[\log D(x)] + \mathbb{E}_{z\sim p_z(z)}[\log(1-D(G(z)))]"}}

Components
Generator

Creates fake images.

Discriminator

Distinguishes real and fake images.

Types of GANs
Type	Description
Vanilla GAN	Basic GAN
DCGAN	Convolutional GAN
Conditional GAN	Conditional generation
CycleGAN	Image translation
StyleGAN	High quality face generation
Difference Between CNN and DCGAN
CNN	DCGAN
Classification	Image generation
Single network	Two competing networks
Predicts labels	Generates images
Applications
Image Generation
Deepfake Creation
Medical Imaging
Data Augmentation
Art Generation
Advantages
Realistic image generation
Learns image features automatically
Powerful generative capability
Limitations
Difficult training
Mode collapse issue
Requires high computation
Viva Questions
What is GAN?

Generative Adversarial Network.

What is mode collapse?

Generator produces similar outputs repeatedly.

What is discriminator?

Network that detects fake images.

EXPERIMENT 6
Transfer Learning using VGG16
Aim

To explore transfer learning using VGG16.

Introduction

Transfer learning uses pre-trained models for new tasks.

VGG16 is a deep CNN trained on ImageNet dataset.

Instead of training from scratch, existing learned features are reused.

Definition

Transfer learning is a technique where knowledge from one model is reused for another related task.

VGG16 Architecture
16 layers
Uses convolution and pooling layers
Fully connected layers at end
Types of Transfer Learning
Type	Description
Feature Extraction	Freeze layers
Fine Tuning	Retrain some layers
Domain Adaptation	Different datasets
Difference Between Training from Scratch and Transfer Learning
Training from Scratch	Transfer Learning
Large data needed	Small data enough
High computation	Faster training
Longer training time	Reduced training time
Applications
Medical Imaging
Object Detection
Face Recognition
Autonomous Vehicles
Image Classification
Advantages
Faster training
Better accuracy
Less data required
Limitations
Pretrained bias possible
Not suitable for all domains
Large model size
Viva Questions
What is transfer learning?

Reusing pretrained model knowledge.

Why freeze layers?

To retain learned features.

What is ImageNet?

Large image dataset used for training.

EXPERIMENT 7
LIME (Local Interpretable Model-Agnostic Explanations)
Aim

To implement and analyze the working of LIME framework.

Introduction

LIME is an explainable AI technique used to understand predictions of machine learning models.

It explains individual predictions locally.

Definition

LIME is a model-agnostic interpretability technique that explains predictions using locally interpretable models.

Key Concepts
Local Explanation
Feature Importance
Model Agnostic
Interpretable Models
Types of Explainable AI
Type	Description
Global Explanation	Entire model explanation
Local Explanation	Single prediction explanation

LIME provides local explanation.

Difference Between LIME and SHAP
LIME	SHAP
Local explanations	Global + local explanations
Faster	More accurate
Simpler	More computationally expensive
Applications
Healthcare AI
Finance
Fraud Detection
Recommendation Systems
Responsible AI
Advantages
Model independent
Easy interpretation
Helps debugging models
Limitations
Local explanation only
Different runs may vary
Approximation errors
Viva Questions
What is explainable AI?

AI whose decisions can be understood by humans.

What does model-agnostic mean?

Works with any machine learning model.

Why is LIME used?

To understand predictions.